In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os

print("EDA environment ready..")

EDA environment ready..


In [26]:
DB_USER = os.getenv("PG_USER")
DB_PASSWORD = os.getenv("PG_PASS")
DB_HOST = os.getenv("PG_HOST")
DB_PORT = os.getenv("PG_PORT", "5432")
DB_NAME = os.getenv("PG_DB")

# Build the PostgreSQL connection URL.
DATABASE_URL = (
    f"postgresql+psycopg://"
    f"{DB_USER}:{DB_PASSWORD}@"
    f"{DB_HOST}:{DB_PORT}/"
    f"{DB_NAME}"
)

# Create the database engine.
engine = create_engine(DATABASE_URL)

print("PostgreSQL engine created.")

PostgreSQL engine created.


In [19]:
# Test that the notebook can actually communicate with PostgreSQL.
with engine.connect() as connection:
    result = connection.exec_driver_sql("SELECT version();")
    print(result.fetchone()[0])

PostgreSQL 18.4 (Debian 18.4-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [20]:
invoices = pd.read_sql(
 """
   SELECT *
   FROM invoices
   ORDER BY invoice_date;
 """,
    engine
)


print(f"Loaded{len(invoices):,} invoices.")

invoices.head()

Loaded1,000 invoices.


,id,invoice_number,supplier_id,customer_id,invoice_date,currency,subtotal,discount,tax_rate,tax_amount,total,created_at
0,1ba9827b-eab4-4f72-97cb-0b98fde0c2ef,INV-000542,9dadbda9-f8a5-42df-982e-093a2c5fa025,b526f849-25d9-4ac9-b574-6ee6552a638f,2025-08-12 23:10:22.161183+00:00,GBP,2648.75,0.00,13.48,357.14,3005.89,2026-08-12 23:10:21.552532+00:00
1,032e93f6-1961-4d33-86c9-fb988439cab0,INV-000770,9d759c2c-a4ec-4749-8e4d-e35706f96c52,b8a00e4d-e0c9-48bf-a25e-94837cb27528,2025-08-12 23:10:22.436519+00:00,GBP,3440.42,0.00,19.87,683.76,4124.18,2026-08-12 23:10:21.552532+00:00
2,6531c703-6acc-43d5-b2b6-c8bbda798bdf,INV-000876,70fac672-8d03-46f5-89a5-057287c9d942,45c8c038-e1e0-4d8a-806b-cfabea701c1e,2025-08-12 23:10:22.561622+00:00,GBP,2183.90,0.00,20.00,436.78,2620.68,2026-08-12 23:10:21.552532+00:00
3,76dfbffa-db13-4ef9-851a-e9d5815c93e2,INV-000489,fc22b204-de19-42ef-8b36-aa49210915df,fc812513-6ffb-4832-9336-79dcdff7a5b0,2025-08-13 23:10:22.100434+00:00,GBP,2242.66,83.82,20.78,448.53,2607.37,2026-08-12 23:10:21.552532+00:00
4,93a93035-4af3-41b0-a175-323be0fc313b,INV-000003,f47a6c1d-291f-4462-a509-ad3ac22206a5,4112c85a-156a-466a-a29b-b8e7d7e405d7,2025-08-14 23:10:21.561137+00:00,GBP,1277.80,0.00,20.00,255.56,1533.36,2026-08-12 23:10:21.552532+00:00


In [21]:
invoice_items = pd.read_sql(
    """
    SELECT *
    FROM invoice_items;
    """,
    engine,
)

print(f"Loaded {len(invoice_items):,} invoice items.")

# Display the first five items.
invoice_items.head()

Loaded 4,534 invoice items.


,id,invoice_id,product_id,quantity,unit_price,discount,tax_rate,tax_amount,subtotal,total
0,28f5edfe-2a7c-4caf-8df5-cb897e197c3c,deeebb15-597f-4264-8a1f-d18dec103014,55046e6b-3496-401c-bab4-7fe602346687,4.499,54.90,0.0,20.0,49.40,247.00,296.39
1,4d774d9b-80b7-4f22-91e0-4ca23b2606dd,deeebb15-597f-4264-8a1f-d18dec103014,1a9f7b6c-cac8-478b-a215-abb3ad887d8a,19.000,46.07,0.0,20.0,175.07,875.33,1050.40
2,9c41dafa-cb1f-4fc2-bd1b-c0160d8cdeb2,deeebb15-597f-4264-8a1f-d18dec103014,2cb87bde-5628-457f-bb49-a098349eb482,10.000,54.44,0.0,20.0,108.88,544.40,653.28
3,5eef64bd-4940-4606-8b14-6d9808aedb72,deeebb15-597f-4264-8a1f-d18dec103014,645c2a59-cf27-4ada-a410-eddbcfb94a34,16.000,29.61,0.0,20.0,94.75,473.76,568.51
4,042b6ae3-038c-4006-9af9-0acaaa62a0eb,deeebb15-597f-4264-8a1f-d18dec103014,4d56cd80-4aaf-457f-9f44-627ba201d050,11.000,45.59,0.0,20.0,100.30,501.49,601.79


In [22]:
anomaly_labels = pd.read_sql(
"""
  SELECT *
  FROM anomaly_labels;
""",
    engine
)

print(f"Loaded {len(anomaly_labels):,} anomaly labels")

anomaly_labels.head()

Loaded 20 anomaly labels


,id,invoice_id,invoice_number,anomaly_type,is_anomaly,created_at
0,aba1f872-df96-45e9-ba2a-96df0fd1d2f5,45758179-9a09-4651-8324-0d63ce728e66,INV-000026,quantity_spike,True,2026-08-12 23:10:21.552532+00:00
1,4de14aaa-59e7-4b0c-b23e-b5de8383e75b,e331248c-c5b9-456f-bb8a-2d3456f20e9a,INV-000031,price_spike,True,2026-08-12 23:10:21.552532+00:00
2,ef0d31af-bd96-49fa-8118-d91caae896b4,52cb4eb7-adcb-4754-8eed-675aaa8b3abe,INV-000033,price_spike,True,2026-08-12 23:10:21.552532+00:00
3,6f344d5c-4c91-42cd-9de9-8be6cc8da429,83db7adb-ccfa-40c1-a76b-04d9ac475356,INV-000090,price_spike,True,2026-08-12 23:10:21.552532+00:00
4,05834e5f-d05e-4322-9411-ce169a048fab,0083d1ff-65ac-4f2c-80ef-0a512b11da94,INV-000096,price_spike,True,2026-08-12 23:10:21.552532+00:00


In [23]:
print("Invoices:", invoices.shape)
print("Invoice items:", invoice_items.shape)
print("Anomaly labels:", anomaly_labels.shape)

Invoices: (1000, 12)
Invoice items: (4534, 10)
Anomaly labels: (20, 6)


In [24]:
invoices.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   id              1000 non-null   object             
 1   invoice_number  1000 non-null   str                
 2   supplier_id     1000 non-null   object             
 3   customer_id     1000 non-null   object             
 4   invoice_date    1000 non-null   datetime64[us, UTC]
 5   currency        1000 non-null   str                
 6   subtotal        1000 non-null   float64            
 7   discount        1000 non-null   float64            
 8   tax_rate        1000 non-null   float64            
 9   tax_amount      1000 non-null   float64            
 10  total           1000 non-null   float64            
 11  created_at      1000 non-null   datetime64[us, UTC]
dtypes: datetime64[us, UTC](2), float64(5), object(3), str(2)
memory usage: 93.9+ KB
